# 01 — MURA XR_WRIST data preparation and EDA

This notebook creates patient-safe internal train and validation manifests. It reads only the official MURA training population. The official MURA valid split is deliberately not read here.

Each output row is one X-ray view. study_uid groups all views from one examination, and patient_id is used to prevent leakage.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
pd.set_option("display.max_columns", 20)

## Data location

In [2]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    pass

DATA_ROOT = Path(os.environ.get(
    "MURA_SOURCE_CSV_DIR",
    os.environ.get("MURA_DATA_ROOT", "/content/drive/MyDrive/DATA/MURA-v1.1"),
))
SPLITS_DIR = Path(os.environ.get(
    "MURA_SPLITS_DIR",
    "/content/drive/MyDrive/DATA/WRIST_SPLITS",
))
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

STUDIES_CSV = DATA_ROOT / "train_labeled_studies.csv"
IMAGES_CSV = DATA_ROOT / "train_image_paths.csv"
for required_path in (STUDIES_CSV, IMAGES_CSV):
    assert required_path.exists(), f"Not found: {required_path}"

print({"source_csv_files_found": True, "split_directory_ready": True, "seed": SEED})

{'source_csv_files_found': True, 'split_directory_ready': True, 'seed': 42}


## Create the XR_WRIST view manifest

In [3]:
studies = pd.read_csv(STUDIES_CSV, header=None, names=["study_path", "label_binary"])
images = pd.read_csv(IMAGES_CSV, header=None, names=["file_name"])

studies = studies.loc[studies["study_path"].str.contains("/XR_WRIST/", regex=False)].copy()
images = images.loc[images["file_name"].str.contains("/XR_WRIST/", regex=False)].copy()

def relative_mura_path(value):
    path = str(value).replace("\\", "/")
    return path.split("MURA-v1.1/", 1)[-1].strip("/")

def study_uid_from_image(value):
    return str(Path(relative_mura_path(value)).parent).replace("\\", "/")

def patient_id_from_path(value):
    return next(part for part in relative_mura_path(value).split("/") if part.startswith("patient"))

studies["study_uid"] = studies["study_path"].map(relative_mura_path).str.rstrip("/")
studies["patient_id"] = studies["study_path"].map(patient_id_from_path)

images = images.loc[
    images["file_name"].str.lower().str.endswith((".png", ".jpg", ".jpeg"))
    & ~images["file_name"].str.contains("/__MACOSX/", regex=False)
    & ~images["file_name"].str.split("/").str[-1].str.startswith("._")
].copy()
images["study_uid"] = images["file_name"].map(study_uid_from_image)
images["patient_id"] = images["file_name"].map(patient_id_from_path)

manifest = images.merge(
    studies[["patient_id", "study_uid", "label_binary"]],
    on=["patient_id", "study_uid"],
    how="inner",
    validate="many_to_one",
)
manifest["label"] = np.where(manifest["label_binary"].eq(1), "positive", "negative")
manifest = manifest[["patient_id", "study_uid", "file_name", "label", "label_binary"]]
manifest = manifest.sort_values(["patient_id", "study_uid", "file_name"], kind="stable").reset_index(drop=True)

assert manifest["file_name"].is_unique
assert manifest.groupby("study_uid")["label_binary"].nunique().max() == 1
display(pd.DataFrame([{
    'manifest_rows': len(manifest),
    'unique_images': manifest['file_name'].nunique(),
    'studies': manifest['study_uid'].nunique(),
    'patients': manifest['patient_id'].nunique(),
}]))

,manifest_rows,unique_images,studies,patients
0,9752,9752,3460,3267


## Study-level summary

In [4]:
study_table = (
    manifest.groupby(["patient_id", "study_uid", "label_binary"], as_index=False)
    .agg(num_views=("file_name", "size"))
)

summary = pd.DataFrame([{
    "patients": study_table["patient_id"].nunique(),
    "studies": len(study_table),
    "images": len(manifest),
    "normal_studies": int((study_table["label_binary"] == 0).sum()),
    "abnormal_studies": int((study_table["label_binary"] == 1).sum()),
}])
view_distribution = (
    study_table["num_views"].value_counts().sort_index()
    .rename_axis("num_views").reset_index(name="studies")
)
display(summary)
display(view_distribution)

,patients,studies,images,normal_studies,abnormal_studies
0,3267,3460,9752,2134,1326


,num_views,studies
0,1,276
1,2,661
2,3,2005
3,4,477
4,5,24
5,6,10
6,7,5
7,8,2


## Patient-safe internal split

In [5]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
train_index, validation_index = next(
    splitter.split(study_table, study_table["label_binary"], groups=study_table["patient_id"])
)

train_studies = study_table.iloc[train_index].copy()
validation_studies = study_table.iloc[validation_index].copy()
train_ids = set(train_studies["study_uid"])
validation_ids = set(validation_studies["study_uid"])

assert train_ids.isdisjoint(validation_ids)
assert set(train_studies["patient_id"]).isdisjoint(set(validation_studies["patient_id"]))

train_manifest = manifest.loc[manifest["study_uid"].isin(train_ids)].copy()
validation_manifest = manifest.loc[manifest["study_uid"].isin(validation_ids)].copy()
train_manifest["split"] = "train"
validation_manifest["split"] = "validation"

split_summary = pd.DataFrame([
    {
        "split": "train",
        "patients": train_studies["patient_id"].nunique(),
        "studies": len(train_studies),
        "images": len(train_manifest),
        "abnormal_studies": int(train_studies["label_binary"].sum()),
    },
    {
        "split": "validation",
        "patients": validation_studies["patient_id"].nunique(),
        "studies": len(validation_studies),
        "images": len(validation_manifest),
        "abnormal_studies": int(validation_studies["label_binary"].sum()),
    },
])
display(split_summary)

,split,patients,studies,images,abnormal_studies
0,train,2614,2769,7810,1057
1,validation,653,691,1942,269


## Save reproducible manifests

In [6]:
train_path = SPLITS_DIR / "wrist_internal_train.csv"
validation_path = SPLITS_DIR / "wrist_internal_val.csv"
studies_path = SPLITS_DIR / "wrist_internal_studies.csv"

train_manifest.to_csv(train_path, index=False)
validation_manifest.to_csv(validation_path, index=False)
pd.concat([
    train_studies.assign(split="train"),
    validation_studies.assign(split="validation"),
], ignore_index=True).to_csv(studies_path, index=False)

provenance = {
    "source_population": "official MURA v1.1 train only, XR_WRIST only",
    "official_valid_read": False,
    "split_method": "StratifiedGroupKFold grouped by patient_id",
    "n_splits": 5,
    "selected_fold": 0,
    "seed": SEED,
    "counts": split_summary.to_dict(orient="records"),
}
(SPLITS_DIR / "data_preparation_summary.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)
print({"train_manifest_written": True, "validation_manifest_written": True})

{'train_manifest_written': True, 'validation_manifest_written': True}


## Handoff

Notebook 02 reads these manifests, audits image decoding, and creates clean derivative manifests. This notebook does not resize images, fit a model, or read the official MURA valid split.